<h1 style="padding-top: 25px; padding-bottom: 25px; text-align: left; padding-left: 10px; background-color: #F7F1E8; border-bottom: 4px solid #D52B1E; color: #1E1C24;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/pavlosprotopapas/elelem-chile-2026/main/assets/epoch_tour_mark.png" height="50px"> <a href="https://pavlosprotopapas.github.io/elelem-chile-2026/" target="_blank"><strong><font color="#1E1C24">ELELEM · Epoch Tour Chile 2026 — From LLMs to Agentic Systems</font></strong></a></h1>

# **<font color="#D52B1E">Agents I lab — 🧀 The Cheese Quiz 🧀</font>**

**ELELEM Chile 2026** · Santiago, 22–24 October 2026<br/>
**Day 2**<br/>
**Instructors:** Pavlos Protopapas (Harvard University) · Guillermo Cabrera-Vives (Universidad de Concepción)<br/>

<hr style="height:2pt">

## 📝 Make a Copy to Edit

This notebook is **view-only**. To edit it, follow these steps:

1. Click **File** > **Save a copy in Drive**.
2. Your own editable copy will open in a new tab.

Now you can modify and run the code freely!

# **Introduction: The Cheese Quiz Guild**

Welcome! In this tutorial, you will build and experience the **Cheese Quiz Guild**: a collaborative multi-agent system where specialized AI agents autonomously research, draft, verify, and format high-quality trivia quizzes on any cheese topic.

# **Working with Agents**


<center><img src="https://drive.google.com/uc?export=view&id=1oCldVicFepwdYbPHcD8Ui4VeyQVJYgut" height="400"><center>



To understand how agents work, we will start from a basic LLM and work our way all the way to a complex multi-agent system. Each step will introduce us to more concepts we learnt in lecture and add features so that we get a solid understanding.

We will use the [OpenAI Python library](https://github.com/openai/openai-python?tab=readme-ov-file) that provides convenient access to the OpenAI REST API to all their foundation models.

We will build agents using the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/). The OpenAI Agents SDK enables you to build agentic AI apps in a lightweight, easy-to-use package with very few abstractions.


# **Learning Objectives**

By the end of this tutorial/guided demo, you will be able to:

- **Understand AI Agents:**
  - Compare and contrast the capabilities of an LLM, a Reasoning LLM, and an Agent.
  - Define agents and their building blocks.

- **Managing LLM Inputs:**
  - Understand the various inputs that affect LLM performance.

- **Tools & MCP (Model Context Protocol):**
  - Build a simple tool and make an agent use it.
  - Use prebuilt MCP tools or build your own.

- **Multi-Agent Systems:**
  - Break a task into smaller micro-agent tasks.
  - Orchestrate a multi-agent system that builds a cheese quiz.


## Notebook Map

We will build the system in stages, beginning with the smallest possible model call and ending with the multi-agent **Cheese Quiz Guild**:

1. **Understanding Agents** — compare a basic LLM, a manually orchestrated reasoning workflow, and an SDK-based agent.
2. **Managing LLM Inputs** — examine how prompts, retrieved knowledge, memory, tool results, and output schemas shape what the model can do.
3. **Tools and MCP** — give agents capabilities through local Python functions and standardized MCP servers.
4. **Multi-Agent Systems** — turn specialized agents into tools and coordinate them with an orchestrator.

**Learning path:**

	simple LLM call → manual reasoning workflow → tool-using agent → context-aware agent → MCP-connected agent → multi-agent guild

The first part focuses on **who controls the workflow**. The context section then focuses on **what information the model receives**. These are complementary views of the same system.

# **Understanding Agents**

**What is an Agent?**<br>
---

An agent is a component that uses an LLM with instructions and tools that can run autonomously to accomplish tasks.


# Prerequisites

Let's make sure we have everything needed. We'll need to have an OpenAI API key for some parts of this tutorial.

Before we can start using the [OpenAI API](https://openai.com/blog/openai-api), we'll need to sign up for an API key from OpenAI. We can do this by visiting the [OpenAI API Keys](https://platform.openai.com/api-keys) page and creating a new API key.

In [57]:
# @title Install Packages
# Installing necessary libraries
!pip -q install fastmcp openai-agents > /dev/null 2>&1
#%pip install openai-agents > /dev/null 2>&1

In [ ]:
# @title Imports
import os
import asyncio
import html
import json, random, time, threading
import openai
from openai import OpenAI
from pathlib import Path
from typing import Dict, Any
from dataclasses import dataclass
from typing import List, Optional, Dict, Literal
from jsonschema import validate as js_validate, ValidationError
from fastmcp import FastMCP
from contextlib import AsyncExitStack
from pydantic import BaseModel, Field, ValidationError
from agents import (
    Agent,
    Runner,
    WebSearchTool,
    function_tool,
    SQLiteSession,
    ModelSettings,
    FunctionTool
)
from agents.agent_output import AgentOutputSchemaBase
from agents.mcp import MCPServerStreamableHttp
from agents.exceptions import ModelBehaviorError
from agents.items import ToolCallItem, ToolCallOutputItem

from IPython.display import HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("openai.agents").setLevel(logging.ERROR)

In [ ]:
# @title Set Up OpenAI Key
# Using Google Colab's Secrets feature 🔑
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
import os
os.environ['OPENAI_API_KEY'] = api_key

#Proper client initialization
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

In [58]:
# @title Download
# Download files
!gdown 1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh # https://drive.google.com/file/d/1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh/view?usp=drive_link
!gdown 1OykEQrwa5cnZgPIVrOdnDeq3txots3H0 # https://drive.google.com/file/d/1OykEQrwa5cnZgPIVrOdnDeq3txots3H0/view?usp=drive_link

Downloading...
From: https://drive.google.com/uc?id=1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh
To: /content/quiz-schema.json
100% 2.10k/2.10k [00:00<00:00, 8.24MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OykEQrwa5cnZgPIVrOdnDeq3txots3H0
To: /content/quiz-ui.html
100% 10.8k/10.8k [00:00<00:00, 65.7MB/s]


---

# **LLM to Agent**

To understand how an Agent works let us use a sample cheese question: **What's the best cheese for a grilled cheese sandwich?**

We will ask this same question to a base LLM, Reasoning LLM, and an Agent.

Compare the responses in each step to understand the concepts.

In [60]:
# Define the LLM model and question
model = "gpt-4o"
user_prompt = "What's the best cheese for a grilled cheese sandwich?"

## **Base LLM**

First we ask the question to an LLM directly with no other bells and whistles.

In [67]:
# @title Define BaseLLM
class BaseLLM:
    """Demonstrates basic LLM functionality - single call, direct response"""

    def __init__(self, model="gpt-4o"):
        self.model = model

    def generate(self, user_prompt: str, max_tokens=1000):
        """Single forward pass through LLM"""
        print("🧠 BaseLLM: Making single API call...")
        start_time = time.time()
        response = client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": user_prompt}],
            max_tokens=max_tokens,
            temperature = 0.2
        )
        end_time = time.time()

        return {
          "content": response.choices[0].message.content,
          "tokens_used": response.usage.total_tokens,
          "response_time": end_time - start_time
      }

In [68]:
# Initialize
base_llm = BaseLLM(model)
llm_result = base_llm.generate(user_prompt)

print("\n" + "="*50)
print("BASE LLM:",model)
print("="*50)
print(f"Response:")
display(Markdown(llm_result['content']))
print(f"Tokens: {llm_result['tokens_used']}, Time: {llm_result['response_time']:.2f}s")

🧠 BaseLLM: Making single API call...

BASE LLM: gpt-4o
Response:


The best cheese for a grilled cheese sandwich often depends on personal preference, but some popular choices include:

1. **American Cheese**: Known for its excellent melting properties and creamy texture, American cheese is a classic choice for a traditional grilled cheese.

2. **Cheddar**: Offers a sharp, tangy flavor and melts well. Aged cheddar can add more depth to the sandwich.

3. **Gruyère**: This Swiss cheese has a nutty, slightly sweet flavor and melts beautifully, making it a great choice for a more gourmet grilled cheese.

4. **Fontina**: Known for its buttery and slightly nutty taste, Fontina melts smoothly and adds a rich flavor.

5. **Monterey Jack**: Mild and creamy, this cheese melts easily and can be paired with other cheeses for added complexity.

6. **Mozzarella**: Offers a mild flavor and excellent stretchiness, making it a good option for those who enjoy a gooey texture.

7. **Provolone**: With a mild, slightly tangy flavor, provolone melts well and can be combined with other cheeses for a more complex taste.

For the best results, you can also mix different cheeses to balance flavors and textures. Adding a touch of butter or mayonnaise to the outside of the bread before grilling can enhance the crispiness and flavor of the sandwich.

Tokens: 293, Time: 2.81s


## **Reasoning LLM**

<img src="https://drive.google.com/uc?export=view&id=1Us_IdQPDZVH8I6ZErXzENItd94KrfNS8" height="400">


Next we build a Reasoning LLM that does the following:
* Step 1: Analyze the type of query and create a reasoning plan
* Step 2: Break down into specific sub-questions
* Step 3: Answer each sub-question specifically
* Step 4: Verify accuracy and completeness
* Step 5: Combine all verified answers into a structured response
* Orchestrate Steps 1–5 to get the final answer

In [ ]:
# @title Define ReasoningLLM
class ReasoningLLM:
    """Demonstrates reasoning framework - multiple orchestrated calls"""

    def __init__(self, base_llm: BaseLLM):
        self.base_llm = base_llm
        self.conversation_history = []

    def analyze_query(self, query: str) -> Dict[str, Any]:
        """Step 1: Analyze the type of query and create a reasoning plan"""
        analysis_prompt = f"""
        Analyze this query and determine the best approach to answer it comprehensively:
        Query: "{query}"

        Respond with:
        1. Query type (factual, process, comparison, etc.)
        2. Key components to address
        3. Suggested breakdown steps

        Keep response concise and structured.
        """

        print("🔧 ReasoningLLM: Step 1 - Analyzing query...")
        response = self.base_llm.generate(analysis_prompt)
        return {"analysis": response["content"], "tokens": response["tokens_used"]}

    def decompose_problem(self, query: str, analysis: str) -> List[str]:
        """Step 2: Break down into specific sub-questions"""
        decomposition_prompt = f"""
        Based on this analysis: {analysis}

        Break down the query "{query}" into 3-4 specific sub-questions that, when answered together,
        will provide a comprehensive response. List only the questions, one per line.
        """

        print("🔧 ReasoningLLM: Step 2 - Decomposing problem...")
        response = self.base_llm.generate(decomposition_prompt)

        # Extract questions from response
        questions = [q.strip() for q in response['content'].split('\n') if q.strip() and '?' in q]
        return questions[:4]  # Limit to 4 questions

    def answer_sub_question(self, question: str) -> str:
        """Step 3: Answer each sub-question specifically"""
        focused_prompt = f"""
        Answer this specific question about cheese making with precise, technical details:
        {question}

        Provide a clear, factual answer focusing only on this aspect.
        """

        print(f"🔧 ReasoningLLM: Step 3 - Answering: {question[:50]}...")
        response = self.base_llm.generate(focused_prompt)
        return response["content"]

    def verify_response(self, question: str, answer: str) -> Dict[str, Any]:
        """Step 4: Verify accuracy and completeness"""
        verification_prompt = f"""
        Question: {question}
        Answer: {answer}

        Evaluate this answer on a scale of 1-10 for:
        1. Accuracy
        2. Completeness
        3. Clarity

        Respond with just three numbers and any critical missing information.
        """

        print("🔧 ReasoningLLM: Step 4 - Verifying response...")
        response = self.base_llm.generate(verification_prompt)
        return {"verification": response["content"]}

    def synthesize_final_response(self, query: str, qa_pairs: List[tuple]) -> str:
        """Step 5: Combine all verified answers into a structured response"""
        synthesis_prompt = f"""
        Original question: {query}

        Sub-questions and answers with verifications:
        {chr(10).join([
            f"Q: {qa['question']}\nA: {qa['answer']}\nVerification: {qa['verification']}\n"
            for qa in qa_pairs
        ])}

        Synthesize these into a well-structured, comprehensive answer to the original question.
        Use clear steps/sections and ensure logical flow.
        """

        print("🔧 ReasoningLLM: Step 5 - Synthesizing final response...")
        response = self.base_llm.generate(synthesis_prompt)
        return response["content"]

    def reason(self, query: str) -> Dict[str, Any]:
        """Main reasoning orchestration method"""
        print(f"\n🔧 ReasoningLLM: Processing '{query}'")
        start_time = time.time()

        # Step 1: Analyze query
        analysis = self.analyze_query(query)

        # Step 2: Decompose into sub-questions
        sub_questions = self.decompose_problem(query, analysis['analysis'])

        # Step 3 & 4: Answer and verify each sub-question
        qa_pairs = []
        total_tokens = analysis['tokens']

        for question in sub_questions:
            answer = self.answer_sub_question(question)
            verification = self.verify_response(question, answer)
            qa_pairs.append({
                "question": question,
                "answer": answer,
                "verification": verification["verification"]
            })
            total_tokens += 100  # Approximate token usage

        # Step 5: Synthesize final response
        final_response = self.synthesize_final_response(query, qa_pairs)

        end_time = time.time()

        return {
            "response": final_response,
            "reasoning_steps": {
                "analysis": analysis['analysis'],
                "sub_questions": sub_questions,
                "qa_pairs": qa_pairs
            },
            "total_tokens": total_tokens,
            "processing_time": end_time - start_time
        }

In [69]:
# Initialize
base_llm = BaseLLM(model)
reasoning_llm = ReasoningLLM(base_llm)

print("\n" + "="*50)
print("REASONING LLM:",model)
print("="*50)
reasoning_result = reasoning_llm.reason(user_prompt)
print(f"Final Response:")
display(Markdown(reasoning_result['response']))
print(f"Sub-questions explored: {len(reasoning_result['reasoning_steps']['sub_questions'])}")
print(f"Total tokens: {reasoning_result['total_tokens']}, Time: {reasoning_result['processing_time']:.2f}s")


REASONING LLM: gpt-4o

🔧 ReasoningLLM: Processing 'What's the best cheese for a grilled cheese sandwich?'
🔧 ReasoningLLM: Step 1 - Analyzing query...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 2 - Decomposing problem...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 1. What qualities make a cheese ideal for use in a...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 4 - Verifying response...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 2. What are the most popular cheese options common...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 4 - Verifying response...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 3. What are the pros and cons of different cheese ...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 4 - Verifying response...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 4. How do personal taste preferences and dietary r...
🧠 Ba

### The Best Cheese for a Grilled Cheese Sandwich

Creating the perfect grilled cheese sandwich involves selecting the right cheese that balances flavor, texture, and meltability. Here’s a comprehensive guide to choosing the best cheese for your grilled cheese sandwich, considering various factors such as cheese qualities, popular options, pros and cons, and personal preferences.

#### 1. Key Qualities of Ideal Cheese for Grilled Cheese

To achieve a delicious grilled cheese sandwich, the cheese should have the following qualities:

- **Meltability**: The cheese should melt smoothly and evenly. Cheeses with higher moisture and fat content, such as American, cheddar, and mozzarella, are known for their excellent melting properties.
  
- **Flavor Profile**: The cheese should offer a balanced flavor that complements the bread and any additional ingredients. Cheddar provides a sharp, tangy taste, while Gruyère offers a nutty, slightly sweet flavor.

- **Texture**: A creamy, gooey texture is desirable. Semi-soft to soft cheeses like fontina and Havarti are known for their smooth consistency when melted.

- **Stretchability**: For those who enjoy a stretchy cheese pull, mozzarella is ideal due to its high elasticity when melted.

- **Browning Ability**: The cheese should develop a golden-brown crust when grilled, achieved with cheeses that have a good balance of proteins and sugars.

#### 2. Popular Cheese Options for Grilled Cheese

Several cheeses are popular choices for grilled cheese sandwiches due to their melting properties and flavors:

- **Cheddar**: Offers a sharp, tangy flavor and excellent melting properties. Available in various intensities from mild to extra sharp.

- **American Cheese**: Known for its smooth, creamy texture and consistent meltability, providing a mild flavor that complements the bread.

- **Swiss Cheese**: With a nutty and slightly sweet flavor, Swiss cheese melts well and adds a distinctive taste.

- **Monterey Jack**: Known for its mild flavor and excellent melting characteristics, often used alone or blended with other cheeses.

- **Mozzarella**: Offers a stretchy, gooey texture when melted, ideal for those who enjoy a stringy cheese pull.

- **Provolone**: This Italian cheese has a mild, slightly tangy flavor and melts smoothly.

#### 3. Pros and Cons of Different Cheese Types

Each cheese type brings unique characteristics to a grilled cheese sandwich. Here are some pros and cons:

- **Cheddar**: Sharp flavor and good melting, but older cheddars can separate into oil and solids.

- **American Cheese**: Excellent melting and consistency, but milder flavor and may contain artificial ingredients.

- **Mozzarella**: Stretchy melt and creamy texture, but mild flavor and high moisture can make the sandwich soggy.

- **Swiss Cheese**: Nutty flavor and smooth melt, but distinct flavor may not appeal to everyone.

- **Gouda**: Rich, buttery flavor and smooth melt, but aged Gouda can be too firm and oily.

- **Provolone**: Mild flavor and even melt, but can become rubbery if not melted properly.

- **Brie**: Rich, creamy flavor and luscious melt, but the rind may not melt well and can alter texture.

#### 4. Personal Preferences and Dietary Considerations

Personal taste preferences and dietary restrictions significantly influence cheese choice:

- **Flavor Preferences**: Some prefer mild cheeses like mozzarella, while others opt for sharper flavors like aged cheddar.

- **Texture and Meltability**: Preferences for smooth, creamy textures might lead to choices like fontina or provolone.

- **Dietary Restrictions**: Lactose-intolerant individuals might choose lactose-free or aged cheeses. Vegans can opt for plant-based cheese alternatives. Health-conscious individuals might select reduced-fat or low-sodium options.

- **Cultural or Regional Preferences**: Different cultures or regions may favor specific cheeses, adding a unique twist to the classic grilled cheese.

In conclusion, the best cheese for a grilled cheese sandwich depends on a combination of meltability, flavor, texture, and personal preferences. By understanding these factors, you can create a grilled cheese sandwich that satisfies your taste and dietary needs.

Sub-questions explored: 4
Total tokens: 701, Time: 39.34s


In [70]:
steps = reasoning_result["reasoning_steps"]

display(Markdown("## Analysis\n" + steps["analysis"]))

for i, qa in enumerate(steps["qa_pairs"], 1):
    display(Markdown(
        f"## Sub-question {i}: {qa['question']}\n\n"
        f"### Answer\n{qa['answer']}\n\n"
        f"### Verification\n{qa['verification']}"
    ))

display(Markdown(
    "## Final synthesis\n" + reasoning_result["response"]
))

## Analysis
1. Query Type:
   - Comparison

2. Key Components to Address:
   - Definition of what makes a cheese suitable for grilled cheese sandwiches (e.g., melting properties, flavor profile).
   - Popular cheese options commonly used for grilled cheese.
   - Pros and cons of different cheese types in the context of grilled cheese sandwiches.
   - Personal preference and variations based on taste.

3. Suggested Breakdown Steps:
   - **Define Criteria**: Explain the qualities that make a cheese ideal for grilled cheese, such as good melting ability and complementary flavor.
   - **List Popular Cheeses**: Identify and describe popular cheeses used in grilled cheese sandwiches, such as American, cheddar, mozzarella, and Swiss.
   - **Compare Cheeses**: Discuss the advantages and disadvantages of each cheese option, focusing on texture, flavor, and melting characteristics.
   - **Consider Personal Preferences**: Highlight how individual taste preferences and dietary restrictions might influence the choice of cheese.
   - **Suggest Combinations**: Offer suggestions for cheese combinations that might enhance the flavor and texture of a grilled cheese sandwich.

## Sub-question 1: 1. What qualities make a cheese ideal for use in a grilled cheese sandwich?

### Answer
To create an ideal grilled cheese sandwich, the cheese used should possess specific qualities that enhance melting, flavor, and texture. Here are the key attributes:

1. **Meltability**: The cheese should have a good melting quality, which is influenced by its moisture content and fat composition. Cheeses with higher moisture and fat content tend to melt more smoothly and evenly. Examples include American, cheddar, and mozzarella.

2. **Flavor Profile**: The cheese should have a balanced flavor that complements the bread and any additional ingredients. It should be flavorful enough to stand out but not overpowering. Cheddar, for instance, offers a sharp, tangy taste, while Gruyère provides a nutty, slightly sweet flavor.

3. **Texture**: The cheese should create a creamy, gooey texture when melted. This is often achieved with semi-soft to soft cheeses that have a smooth consistency. Cheeses like fontina and Havarti are known for their creamy melt.

4. **Stretchability**: For those who enjoy a stretchy cheese pull, cheeses like mozzarella are ideal due to their high elasticity when melted.

5. **Browning Ability**: The cheese should be able to develop a desirable golden-brown crust when grilled. This is often achieved with cheeses that have a good balance of proteins and sugars, which promote the Maillard reaction.

By selecting cheeses that exhibit these qualities, you can create a grilled cheese sandwich with optimal melt, flavor, and texture.

### Verification
9, 9, 9

Critical missing information: The answer could include examples of cheeses that do not work well in grilled cheese sandwiches due to poor meltability or overpowering flavors, such as hard cheeses like Parmesan or crumbly cheeses like feta.

## Sub-question 2: 2. What are the most popular cheese options commonly used for grilled cheese sandwiches?

### Answer
The most popular cheese options commonly used for grilled cheese sandwiches are:

1. **Cheddar**: Known for its sharp, tangy flavor and excellent melting properties, cheddar is a classic choice for grilled cheese. It ranges from mild to extra sharp, allowing for customization of flavor intensity.

2. **American Cheese**: This processed cheese is favored for its smooth, creamy texture and consistent meltability. It provides a mild flavor that complements the buttery bread.

3. **Swiss Cheese**: With its nutty and slightly sweet flavor, Swiss cheese melts well and adds a distinctive taste to grilled cheese sandwiches.

4. **Monterey Jack**: Known for its mild flavor and excellent melting characteristics, Monterey Jack is often used alone or blended with other cheeses for a creamy texture.

5. **Mozzarella**: While not as flavorful as other options, mozzarella offers a stretchy, gooey texture when melted, making it a popular choice for those who enjoy a stringy cheese pull.

6. **Provolone**: This Italian cheese has a mild, slightly tangy flavor and melts smoothly, making it a versatile option for grilled cheese.

These cheeses are chosen for their melting properties, flavor profiles, and ability to complement the buttery, toasted bread typically used in grilled cheese sandwiches.

### Verification
10, 10, 10

## Sub-question 3: 3. What are the pros and cons of different cheese types when used in grilled cheese sandwiches?

### Answer
When considering different cheese types for grilled cheese sandwiches, it's important to evaluate their melting properties, flavor profiles, and textures. Here are the pros and cons of some popular cheese options:

### Cheddar
- **Pros:**
  - **Flavor:** Offers a sharp, tangy flavor that is a classic choice for grilled cheese.
  - **Melting:** Medium to sharp cheddar melts well, providing a gooey texture.
- **Cons:**
  - **Oil Separation:** Older, sharper cheddars can sometimes separate into oil and solids when melted.
  - **Texture:** Can become greasy if not balanced with other ingredients.

### American Cheese
- **Pros:**
  - **Melting:** Known for its excellent melting properties, creating a smooth and creamy texture.
  - **Consistency:** Provides a uniform melt and flavor.
- **Cons:**
  - **Flavor:** Has a milder taste, which might not be as flavorful as other cheeses.
  - **Artificial Ingredients:** Often contains emulsifiers and preservatives.

### Mozzarella
- **Pros:**
  - **Melting:** Offers a stretchy, gooey melt, especially fresh mozzarella.
  - **Texture:** Provides a creamy texture when melted.
- **Cons:**
  - **Flavor:** Mild flavor may require additional seasoning or pairing with stronger cheeses.
  - **Moisture Content:** High moisture can make the sandwich soggy if not properly drained.

### Swiss Cheese
- **Pros:**
  - **Flavor:** Nutty and slightly sweet, adding complexity to the sandwich.
  - **Melting:** Melts well, providing a smooth texture.
- **Cons:**
  - **Flavor:** The distinct flavor may not appeal to everyone.
  - **Texture:** Can become rubbery if overcooked.

### Gouda
- **Pros:**
  - **Flavor:** Offers a rich, buttery flavor that enhances the sandwich.
  - **Melting:** Melts smoothly, especially younger varieties.
- **Cons:**
  - **Aging:** Aged Gouda can be too firm and may not melt as well.
  - **Oiliness:** Can become oily when melted.

### Provolone
- **Pros:**
  - **Flavor:** Mild and slightly tangy, complementing other ingredients.
  - **Melting:** Melts evenly, providing a creamy texture.
- **Cons:**
  - **Flavor:** May be too mild for those seeking a stronger cheese flavor.
  - **Texture:** Can become rubbery if not melted properly.

### Brie
- **Pros:**
  - **Flavor:** Rich and creamy with a buttery taste, adding sophistication.
  - **Melting:** Melts into a luscious, creamy texture.
- **Cons:**
  - **Rind:** The rind may not melt well and can alter the texture.
  - **Flavor:** The strong flavor may not be suitable for all palates.

Each cheese type brings unique characteristics to a grilled cheese sandwich, and often, a blend of cheeses can provide the best balance of flavor and texture.

### Verification
9, 9, 10

Critical missing information: The answer could include more details on how to balance cheeses with other ingredients to mitigate cons like oiliness or sogginess. Additionally, it could mention other popular cheeses like Pepper Jack or Havarti for a more comprehensive overview.

## Sub-question 4: 4. How do personal taste preferences and dietary restrictions influence the choice of cheese for a grilled cheese sandwich?

### Answer
Personal taste preferences and dietary restrictions significantly influence the choice of cheese for a grilled cheese sandwich in several ways:

1. **Flavor Profile**: Personal taste preferences dictate the flavor profile of the cheese selected. Some individuals may prefer mild cheeses like mozzarella or Monterey Jack for their subtle taste, while others might opt for sharper, more robust flavors like aged cheddar or gouda. The choice can also be influenced by the desired meltability and texture, with cheeses like American cheese being favored for their smooth melting properties.

2. **Texture and Meltability**: The texture and meltability of cheese are crucial for a grilled cheese sandwich. Personal preferences might lead someone to choose a cheese that melts smoothly and evenly, such as fontina or provolone, to achieve a creamy consistency. Conversely, those who prefer a firmer texture might choose a cheese that retains some structure when melted.

3. **Dietary Restrictions**: Dietary restrictions play a critical role in cheese selection. For individuals with lactose intolerance, lactose-free cheeses or aged cheeses like parmesan, which have lower lactose content, might be preferred. Those following a vegan diet would opt for plant-based cheese alternatives made from nuts, soy, or coconut oil. Additionally, individuals with allergies or specific dietary needs might choose cheeses that are free from certain additives or preservatives.

4. **Nutritional Considerations**: Health-conscious individuals might select cheeses based on their nutritional content, such as reduced-fat or low-sodium options. For example, part-skim mozzarella or reduced-fat cheddar might be chosen to lower calorie and fat intake while still enjoying the flavor and texture of cheese.

In summary, the choice of cheese for a grilled cheese sandwich is influenced by a combination of personal taste preferences, desired texture and meltability, and dietary restrictions, ensuring that the selected cheese aligns with individual flavor desires and health considerations.

### Verification
9, 9, 9

Critical missing information: The answer could include more examples of specific cheeses that cater to dietary restrictions, such as specific brands of vegan cheese or lactose-free options. Additionally, mentioning cultural or regional preferences could enhance completeness.

## Final synthesis
### The Best Cheese for a Grilled Cheese Sandwich

Creating the perfect grilled cheese sandwich involves selecting the right cheese that balances flavor, texture, and meltability. Here’s a comprehensive guide to choosing the best cheese for your grilled cheese sandwich, considering various factors such as cheese qualities, popular options, pros and cons, and personal preferences.

#### 1. Key Qualities of Ideal Cheese for Grilled Cheese

To achieve a delicious grilled cheese sandwich, the cheese should have the following qualities:

- **Meltability**: The cheese should melt smoothly and evenly. Cheeses with higher moisture and fat content, such as American, cheddar, and mozzarella, are known for their excellent melting properties.
  
- **Flavor Profile**: The cheese should offer a balanced flavor that complements the bread and any additional ingredients. Cheddar provides a sharp, tangy taste, while Gruyère offers a nutty, slightly sweet flavor.

- **Texture**: A creamy, gooey texture is desirable. Semi-soft to soft cheeses like fontina and Havarti are known for their smooth consistency when melted.

- **Stretchability**: For those who enjoy a stretchy cheese pull, mozzarella is ideal due to its high elasticity when melted.

- **Browning Ability**: The cheese should develop a golden-brown crust when grilled, achieved with cheeses that have a good balance of proteins and sugars.

#### 2. Popular Cheese Options for Grilled Cheese

Several cheeses are popular choices for grilled cheese sandwiches due to their melting properties and flavors:

- **Cheddar**: Offers a sharp, tangy flavor and excellent melting properties. Available in various intensities from mild to extra sharp.

- **American Cheese**: Known for its smooth, creamy texture and consistent meltability, providing a mild flavor that complements the bread.

- **Swiss Cheese**: With a nutty and slightly sweet flavor, Swiss cheese melts well and adds a distinctive taste.

- **Monterey Jack**: Known for its mild flavor and excellent melting characteristics, often used alone or blended with other cheeses.

- **Mozzarella**: Offers a stretchy, gooey texture when melted, ideal for those who enjoy a stringy cheese pull.

- **Provolone**: This Italian cheese has a mild, slightly tangy flavor and melts smoothly.

#### 3. Pros and Cons of Different Cheese Types

Each cheese type brings unique characteristics to a grilled cheese sandwich. Here are some pros and cons:

- **Cheddar**: Sharp flavor and good melting, but older cheddars can separate into oil and solids.

- **American Cheese**: Excellent melting and consistency, but milder flavor and may contain artificial ingredients.

- **Mozzarella**: Stretchy melt and creamy texture, but mild flavor and high moisture can make the sandwich soggy.

- **Swiss Cheese**: Nutty flavor and smooth melt, but distinct flavor may not appeal to everyone.

- **Gouda**: Rich, buttery flavor and smooth melt, but aged Gouda can be too firm and oily.

- **Provolone**: Mild flavor and even melt, but can become rubbery if not melted properly.

- **Brie**: Rich, creamy flavor and luscious melt, but the rind may not melt well and can alter texture.

#### 4. Personal Preferences and Dietary Considerations

Personal taste preferences and dietary restrictions significantly influence cheese choice:

- **Flavor Preferences**: Some prefer mild cheeses like mozzarella, while others opt for sharper flavors like aged cheddar.

- **Texture and Meltability**: Preferences for smooth, creamy textures might lead to choices like fontina or provolone.

- **Dietary Restrictions**: Lactose-intolerant individuals might choose lactose-free or aged cheeses. Vegans can opt for plant-based cheese alternatives. Health-conscious individuals might select reduced-fat or low-sodium options.

- **Cultural or Regional Preferences**: Different cultures or regions may favor specific cheeses, adding a unique twist to the classic grilled cheese.

In conclusion, the best cheese for a grilled cheese sandwich depends on a combination of meltability, flavor, texture, and personal preferences. By understanding these factors, you can create a grilled cheese sandwich that satisfies your taste and dietary needs.

## **Agent**

An agent is a component that uses an LLM with instructions and tools that can run autonomously to accomplish tasks.

<center><img src="https://drive.google.com/uc?export=view&id=1TgpuVStg8ChZBMubgd-_Ty2wrMdGWcOY" height="400"><center>

Next we build an Agent that does the following:
* Perform the same reasoning strategy as our reasoning LLM
* Add a web search tool so the agent has the capability to fact-check using the internet

In [71]:
# @title Define SimpleAgent
class SimpleAgent:
    """Demonstrates agent functionality - LLM, instructions, tool usage (research-oriented)"""

    def __init__(
        self,
        model: str = "gpt-4o",
        user_location: Dict[str, Any] | None = None,
        search_context_size: str = "medium",          # "low" | "medium" | "high"
        instructions: str | None = None,
    ):
        # Configure the web search tool
        web_tool = WebSearchTool(
            user_location=user_location,
            search_context_size=search_context_size
        )

        # Instructions similar to Reasoning LLM
        if instructions is None:
            instructions = (
                "You are a meticulous research agent. "
                "Before answering, outline a brief plan with 3–4 sub-questions. "
                "Use the web_search tool for up-to-date or factual details. "
                "Perform at least TWO web_search calls across DIFFERENT reputable domains. "
                "Cross-check key facts (dates, figures, definitions). "
                "Then write a clear, structured answer with sections and practical recommendations. "
                "Finish with a 'Sources' list (title + URL) covering the items you used."
            )

        self.agent = Agent(
            name="SimpleAgent",
            instructions=instructions,
            model=model,
            tools=[web_tool],
        )

    def _research_prompt(self, query: str) -> str:
        return f"""
Task: Answer the user's question comprehensively.

User question: {query}

Workflow you must follow:
1) **Plan**: List 3–4 concise sub-questions you will answer.
2) **Research**: Use `web_search` at least twice across different domains to gather facts.
3) **Synthesize**: Combine findings into a cohesive answer that is better than a generic LLM response.
4) **Verify**: Re-check critical claims and resolve conflicts if sources disagree.
5) **Present**: Write the final answer with these sections:
   - Summary (2–4 sentences)
   - Key Criteria / How to Choose
   - Top Recommendations (with brief justifications)
   - Smart Variations or Blends (if applicable)
   - Practical Tips (actionable steps)
   - Sources (bullet list: Title — URL)

Output format: Markdown.
If uncertainty remains, say what is uncertain and why.
"""

    async def generate(self, query: str) -> Dict[str, Any]:
        print(f"\n🤖 SimpleAgent: Running with web_search for '{query}'")
        start = time.time()

        # 🔧 pass the guided research prompt
        result = await Runner.run(self.agent, input=self._research_prompt(query))

        end = time.time()
        return {
            "response": result.final_output,
            "processing_time": end - start,
            "tools": self.agent.tools  # available tools (actual usage will show in logs/tracing if enabled)
        }


In [73]:
# Initialize
agent = SimpleAgent(model)

print("\n" + "="*50)
print("AGENT:",model)
print("="*50)
agent_result = await agent.generate(user_prompt)
print(f"Final Response:")
display(Markdown(agent_result['response']))
print(f"Tools used: {len(agent_result['tools'])}")
print(f"Time: {agent_result['processing_time']:.2f}s")


AGENT: gpt-4o

🤖 SimpleAgent: Running with web_search for 'What's the best cheese for a grilled cheese sandwich?'
Final Response:


Here’s the research plan I followed:

**Plan — Key Sub‑questions**
1. What essential qualities make a cheese ideal for grilled‑cheese sandwiches?
2. Which cheeses are most commonly recommended—both as single choices and in blends?
3. What blends or variations elevate both melt and flavor?
4. What practical tips improve grilling technique and result?

---

## Research Findings

### Search 1: "best cheese for grilled cheese sandwich what cheese melts best"
Key sources:
- **KnowTheCheese** emphasizes American cheese as the most reliable single choice for fast, even melting; natural cheeses like young cheddar, Gruyère, and Jack for more flavor, and blending for best results ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).
- **The Sandwich App** underlines that meltability and flavor seldom coexist in one cheese and advises blending a quick melter (like American, Monterey Jack, Havarti) with a flavor-forward cheese (like sharp cheddar or Gruyère) ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).

### Search 2: "best grocery store cheeses for grilled cheese"
Key sources:
- **Eat This, Not That** cites Tillamook Medium Cheddar for balanced melt and flavor; also recommends pairing sharper, aged cheddar with a creamier melter like Havarti or provolone ([eatthis.com](https://www.eatthis.com/best-grocery-store-cheeses-grilled-cheese/?utm_source=openai)).

---

## Synthesis & Verification

Based on multiple reputable and recent sources, these critical points emerge consistently:

- **Meltability** depends on cheese moisture, fat, emulsifiers, and age. Younger, moister cheeses (e.g., American, Monterey Jack, Havarti) melt smoothly. Aged cheeses (sharp cheddar, aged Gouda) offer flavor but risk oiling or graininess unless used sparingly or in a blend ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).
- **Flavor** comes often from harder or aged cheeses. Neither alone can deliver ideal melt and flavor simultaneously; blending is the key strategy ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).
- **Blending** pairs a melter with a flavor cheese—e.g., American or Monterey Jack base plus sharp cheddar or Gruyère for depth. Many sources recommend around a 2:1 ratio (melter to flavor cheese) ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).
- **Single best choices**:
  - **American cheese**: fastest, creamiest melt; safe go-to ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).
  - **Tillamook Medium Cheddar**: great all-around melt plus savory cheddar flavor ([eatthis.com](https://www.eatthis.com/best-grocery-store-cheeses-grilled-cheese/?utm_source=openai)).
  - **Natural options**: Gruyère (nutty), Havarti (smooth, buttery), Fontina, and Monterey Jack are often cited for both melt and character ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).

---

## Final Answer

### Summary  
The best cheese for a grilled‑cheese sandwich balances two things: meltability and flavor. For dependable melty success, American cheese is the classic go-to; for richer taste, pair it—or another good melter—with a sharper cheese in a blend.

---

### Key Criteria / How to Choose  
- **Meltability**: Look for cheese with moderate moisture, fat content, and—ideally—emulsifiers (e.g., American cheese). Younger cheeses melt more evenly; aged ones may split or weep without assistance ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
- **Flavor**: Aged or tangy cheeses (cheddar, Gruyère, aged Gouda) offer complexity but typically need a melting partner ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
- **Timing**: Your cheese should melt within the same timeframe that the bread toasts. If it melts too fast, it may ooze; too slow, and the bread burns ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).

---

### Top Recommendations  
1. **American cheese** — the most reliable for smooth, edge‑to‑edge melting and forgiving timing ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).  
2. **Tillamook Medium Cheddar** — all‑around performer with good melt and satisfying cheddar taste ([eatthis.com](https://www.eatthis.com/best-grocery-store-cheeses-grilled-cheese/?utm_source=openai)).  
3. **Gruyère** — nutty, sophisticated melt; great upgrade for thicker bread and slower heat ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).  
4. **Havarti** — buttery, smooth, and melts beautifully ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
5. **Monterey Jack** — mild, creamy, excellent melt; perfect base for blending ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).

---

### Smart Variations or Blends  
- **Classic blend**: American cheese + sharp cheddar (approx. 2:1 ratio) for both melt and flavor ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
- **Elevated blend**: Gruyère + Fontina or Havarti for a richer, Swiss‑style sandwich ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).  
- **Stretchy combo**: Low‑moisture mozzarella with young provolone or cheddar for pull factor and bold taste ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).

---

### Practical Tips  
- **Grate or thin‑slice cheese** — increases surface area for faster, even melting; avoid pre‑shredded (anti‑caking agents impede melt) ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
- **Bring cheese closer to room temperature** — helps it soften more quickly once the sandwich hits the pan ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
- **Cook low and slow** — use a heavy skillet or cast iron, optionally cover briefly to trap heat and melt before the bread over‑browns ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
- **Spread outside of bread with mayo, not butter** — mayonnaise browns more evenly and crisps reliably ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai)).  
- **Use appropriate bread thickness** — thin white bread melts faster; thick sourdough benefits from richer melt (e.g., Gruyère) ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai)).

---

### Sources  
- Best Cheese for Grilled Cheese: 9 Picks | KnowTheCheese — KnowTheCheese ([knowthecheese.com](https://knowthecheese.com/best-for/grilled-cheese/?utm_source=openai))  
- The Best Cheese for Grilled Cheese — The Sandwich App ([thesandwichapp.com](https://www.thesandwichapp.com/guides/best-cheese-for-grilled-cheese?utm_source=openai))  
- 5 Best Grocery Store Cheeses for Grilled Cheese, According to Chefs — Eat This, Not That ([eatthis.com](https://www.eatthis.com/best-grocery-store-cheeses-grilled-cheese/?utm_source=openai))  
- Best Cheese for Grilled Cheese Sandwich: 9 That Melt Perfect – Gardners Wisconsin Cheese and Sausage ([gardnerswisconsincheese.com](https://www.gardnerswisconsincheese.com/blogs/articles/the-best-cheese-for-a-grilled-cheese-sandwich-9-cheeses-that-melt-like-a-dream?utm_source=openai))  
- Best Cheese for Grilled Cheese: Melt, Flavor, and Sandwich Structure — Sandwich Lovers ([sdlovers.com](https://sdlovers.com/journal/best-cheese-for-grilled-cheese?utm_source=openai))

---

If any uncertainty remains—such as preference for vegan cheeses or specific dietary constraints—I’d be happy to explore further!

Tools used: 1
Time: 13.29s


---

# **Managing LLM Inputs**

<center><img src="https://drive.google.com/uc?export=view&id=1YkMHoyZnJqsLSoBgDrNzQabKT0k5OW6h" height="400"><center>




LLMs are stateless functions. To get the best results, you need to give LLMs the best inputs. Managing LLM inputs (context) is about building:
* The instructions you give to the model. The system prompt & user prompt
* Any documents or external data you retrieve (e.g. RAG)
* Any tool call output
* Any past state or other history (Short-term Memory)
* Any past messages or events from related but separate histories/conversations (Long-term Memory)
* Instructions about what sorts of structured data to output

***Think of model context as the invisible layer that feeds and connects all parts of the agent.***

It manages what information the LLM sees, when it sees it, and how it uses it.
Without the right context, even the best reasoning model cannot plan, adapt, or remember effectively.

We have already seen a basic prompt in the previous section, but here we deliberately build again from the basic prompt. This time, the goal is different: instead of comparing an LLM with an agent, we hold the model and question constant so we can isolate the effect of each additional piece of context. Starting from zero gives us a clean baseline, and then we add one layer at a time.

To understand how model context works let us use the same sample cheese question: **What's the best cheese for a grilled cheese sandwich?**

We will ask this same question to an LLM while incrementally adjusting the context, and compare the results from each step.

* Step 1: Basic prompt
* Step 2: Prompt Engineering
* Step 3: Retrieve Relevant Documents (e.g. RAG)
* Step 4: Adding Short-term Memory
* Step 5: Adding Long-term Memory
* Step 6: Tool Call Outputs
* Step 7: Structured Output

In [ ]:
model = "gpt-4o"
user_prompt = "What's the best cheese for a grilled cheese sandwich?"

In [74]:
# @title Step 1: Basic prompt
print("\n" + "="*70)
print("Step 1: Basic Prompt")
print("="*70)

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 1: Basic Prompt


The best cheese for a grilled cheese sandwich often depends on personal preference, but some popular choices include:

1. **Cheddar**: A classic choice, cheddar melts well and offers a sharp, tangy flavor.
2. **American**: Known for its excellent melting properties and creamy texture, American cheese is a favorite for many.
3. **Gruyère**: This Swiss cheese adds a nutty, slightly sweet flavor and melts beautifully.
4. **Monterey Jack**: Mild and creamy, it melts smoothly and pairs well with other cheeses.
5. **Provolone**: Offers a mild, slightly smoky flavor and melts nicely.
6. **Mozzarella**: Provides a gooey texture and mild flavor, great for mixing with other cheeses.

For a more complex flavor, you can mix different cheeses. For example, combining cheddar with Gruyère or adding a bit of blue cheese for a tangy kick can elevate your grilled cheese sandwich.

In [75]:
# @title Step 2: Prompt Engineering
print("\n" + "="*70)
print("Step 2: With Prompt Engineering")
print("="*70)

# Well-crafted system prompt
system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning."""

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": system_prompt},
        {"role": "system", "content": user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 2: With Prompt Engineering


When crafting the perfect grilled cheese sandwich, the choice of cheese is crucial for achieving that ideal combination of meltability, flavor, and texture. Here are some top recommendations:

1. **American Cheese**:
   - **Melting Properties**: American cheese is known for its excellent melting properties, creating a smooth and creamy texture when heated. It melts at a relatively low temperature, around 130°F (54°C).
   - **Flavor Profile**: It has a mild, slightly tangy flavor that pairs well with various breads and additional ingredients.
   - **Availability**: Widely available in most grocery stores, often found pre-sliced in the dairy section.
   - **Practical Cooking Techniques**: Its uniform melting makes it easy to achieve a consistent texture throughout the sandwich. Simply layer a few slices between your bread of choice and grill until golden brown.

2. **Cheddar Cheese**:
   - **Melting Properties**: Cheddar melts well, especially when aged for a shorter period (mild to medium cheddar). It melts at around 150°F (66°C).
   - **Flavor Profile**: Offers a range from mild to sharp, allowing you to customize the intensity of your sandwich. Sharp cheddar provides a more pronounced flavor.
   - **Availability**: Readily available in blocks, slices, or shredded form in most grocery stores.
   - **Practical Cooking Techniques**: For a smoother melt, grate the cheese before adding it to the sandwich. This helps it melt evenly and quickly.

3. **Gruyère Cheese**:
   - **Melting Properties**: Gruyère is known for its excellent melting qualities, often used in fondue. It melts at around 135°F (57°C).
   - **Flavor Profile**: Offers a nutty, slightly sweet flavor that becomes more pronounced as it ages.
   - **Availability**: Available in most grocery stores, though it may be slightly more expensive than other options.
   - **Practical Cooking Techniques**: Slice thinly or grate to ensure even melting. Gruyère pairs well with additional ingredients like caramelized onions or ham for a more gourmet grilled cheese.

4. **Mozzarella Cheese**:
   - **Melting Properties**: Mozzarella melts beautifully, creating a stretchy, gooey texture. It melts at around 130°F (54°C).
   - **Flavor Profile**: Mild and creamy, allowing it to blend well with other flavors.
   - **Availability**: Easily found in most grocery stores, available in blocks, pre-sliced, or shredded.
   - **Practical Cooking Techniques**: Use fresh mozzarella for a creamier texture or low-moisture mozzarella for a more traditional grilled cheese. Combine with a more flavorful cheese like cheddar for added depth.

5. **Provolone Cheese**:
   - **Melting Properties**: Provolone melts smoothly, making it a great choice for grilled cheese. It melts at around 140°F (60°C).
   - **Flavor Profile**: Offers a mild to slightly tangy flavor, depending on its age.
   - **Availability**: Commonly available in deli sections and pre-packaged slices.
   - **Practical Cooking Techniques**: Layer with other cheeses like mozzarella or cheddar for a complex flavor profile.

For the ultimate grilled cheese experience, consider combining two or more cheeses to balance meltability and flavor. A classic combination is cheddar and mozzarella, which provides both sharpness and creaminess. Remember to use a good quality bread, butter the outside for a crispy crust, and cook on medium-low heat to ensure the cheese melts thoroughly without burning the bread.

In [82]:
# @title Step 3: Retrieve Relevant Documents (e.g. RAG)
# Simulated knowledge base (in real apps, this would come from a vector database)
CHEESE_KNOWLEDGE_BASE = {
    "american_cheese": {
        "melting_temp": "150°F",
        "properties": "Smooth melt, mild flavor, contains emulsifiers for perfect texture",
        "best_for": "Classic grilled cheese, kids' sandwiches"
    },
    "cheddar": {
        "melting_temp": "170°F",
        "properties": "Rich flavor, can separate if overheated, varies by age",
        "best_for": "Flavor-focused sandwiches, adults"
    },
    "gruyere": {
        "melting_temp": "170°F",
        "properties": "Nutty and complex, excellent melt, slightly sweet",
        "best_for": "Gourmet grilled cheese, French onion soup"
    }
}

print("\n" + "="*70)
print("Step 3: With RAG (Retrieved Knowledge)")
print("="*70)

# System prompt
system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning."""

# Inject retrieved knowledge into the prompt
enhanced_user_prompt = f"""Based on this cheese data:
{str(CHEESE_KNOWLEDGE_BASE)}
User question: {user_prompt}"""

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": enhanced_user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 3: With RAG (Retrieved Knowledge)


When choosing the best cheese for a grilled cheese sandwich, consider both the melting properties and the flavor profile you desire.

1. **American Cheese**: This is a classic choice for grilled cheese sandwiches, especially if you're looking for a smooth melt and a mild flavor. American cheese melts at around 150°F, which is lower than other cheeses, making it easy to achieve that gooey texture without much effort. Its emulsifiers ensure a perfect, uniform melt, making it ideal for a straightforward, comforting sandwich. It's also widely available in most grocery stores.

2. **Cheddar**: If you're looking for a more flavor-focused sandwich, cheddar is an excellent option. It melts at around 170°F, so you'll need to be a bit more careful with the heat to prevent it from separating. A medium cheddar offers a good balance between meltability and flavor, while sharp cheddar provides a more intense taste. Cheddar is also readily available and can add a rich, tangy flavor to your sandwich.

3. **Gruyere**: For a gourmet twist, gruyere is an exceptional choice. It melts beautifully at around 170°F and has a nutty, slightly sweet flavor that can elevate your grilled cheese to a more sophisticated level. Gruyere is perfect if you're aiming for a more complex flavor profile, perhaps paired with additional ingredients like caramelized onions or mushrooms.

In summary, for a classic and easy-to-make grilled cheese, go with American cheese. If you want a richer flavor, opt for cheddar. For a gourmet experience, choose gruyere. Each cheese brings its unique qualities to the table, so your choice depends on the flavor and texture you're aiming for.

In [85]:
# @title Step 4: Adding Short-term Memory
print("\n" + "="*70)
print("Step 4: With Conversation State/History (Short-term Memory)")
print("="*70)

system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

Remember the user's preferences and previous questions in this conversation"""

# Build conversation history
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm cooking for my kids who prefer mild flavors"},
    {"role": "assistant", "content": "Great! I'll keep that in mind. Kids often prefer milder, creamier cheeses. What would you like to make?"},
    {"role": "user", "content": "Should I use butter or mayo on the bread?"},
    {"role": "assistant", "content": "For kids, I'd recommend butter on the outside of the bread - it creates a golden, crispy texture they'll love. Mayo works too and browns more evenly, but butter has a more familiar, comforting taste."},
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 4: With Conversation State/History (Short-term Memory)


For a grilled cheese sandwich with mild flavors, I recommend using a combination of mild cheddar and mozzarella. 

- **Mild Cheddar**: It melts well and has a creamy texture with a gentle flavor that kids usually enjoy. It starts melting around 150°F (65°C), making it perfect for a quick sandwich.

- **Mozzarella**: Known for its excellent melting properties, mozzarella adds a gooey texture and a mild, milky taste. It melts at a slightly lower temperature, around 130°F (54°C), which complements the cheddar nicely.

Both cheeses are readily available in most grocery stores. You can use them in equal parts for a balanced flavor and texture. Make sure to shred the cheese for even melting. Cook the sandwich on medium-low heat to ensure the cheese melts thoroughly without burning the bread. Enjoy!

In [86]:
# @title Step 5: Adding Long-term Memory
print("\n" + "="*70)
print("Step 5: With Long-term Memory (Past Sessions)")
print("="*70)

# Information from past conversations (assume stored in a database)
USER_MEMORY = {
    "user_id": "user_123",
    "preferences": {
        "previous_cheese_choice": "mild cheddar (mac and cheese, 2 weeks ago)",
        "dietary_restrictions": "lactose sensitivity",
        "shopping_location": "Whole Foods",
        "budget_preference": "under $20",
        "household": "cooking for kids",
        "health_preference": "Prioritize health-conscious choices; consider portion size, sodium, and saturated fat.",
        "ingredient_preference": (
            "Only recommend cheeses with simple, natural ingredients such as milk, cultures, salt, and enzymes/rennet. "
            "Avoid artificial colors, artificial flavors, artificial preservatives, and processed cheese products with added emulsifying salts. "
            "Check the actual ingredient list; if it is unavailable, do not assume the product meets this preference. "
            "Ingredient requirements take priority over price or melting performance. Do not equate natural ingredients with automatically being healthier."
        )
    }
}

# Load user memory
memory_context = f"""User Profile:
- Previously enjoyed: {USER_MEMORY['preferences']['previous_cheese_choice']}
- Dietary note: {USER_MEMORY['preferences']['dietary_restrictions']}
- Shops at: {USER_MEMORY['preferences']['shopping_location']}
- Budget: {USER_MEMORY['preferences']['budget_preference']}
- Household: {USER_MEMORY['preferences']['household']}
- Health priorities: {USER_MEMORY['preferences']['health_preference']}
- Ingredient requirements: {USER_MEMORY['preferences']['ingredient_preference']}"""

# System prompt
system_prompt = f"""You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

{memory_context}

Use this information to personalize your recommendations."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm cooking for my kids who prefer mild flavors"},
    {"role": "assistant", "content": "Great! I'll keep that in mind. Kids often prefer milder, creamier cheeses. What would you like to make?"},
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.2,
)
display(Markdown(response.choices[0].message.content))


Step 5: With Long-term Memory (Past Sessions)


For a grilled cheese sandwich that caters to mild flavors and your dietary preferences, I recommend using **Havarti** cheese. Here's why:

1. **Flavor Profile**: Havarti is a semi-soft cheese with a mild, buttery taste that kids typically enjoy. It has a creamy texture that complements the classic grilled cheese.

2. **Melting Properties**: Havarti melts beautifully, creating a smooth and gooey texture that's perfect for grilled cheese. It melts at a relatively low temperature, so you can achieve that perfect melt without overcooking the bread.

3. **Availability**: You can find Havarti at Whole Foods, and it often comes in blocks or pre-sliced, making it convenient for sandwiches.

4. **Ingredients**: Look for Havarti with simple ingredients like milk, cultures, salt, and enzymes. Brands like Roth or Arla often offer natural options without artificial additives.

5. **Health Considerations**: Havarti is generally lower in sodium compared to some other cheeses, and you can control portion sizes to manage saturated fat intake. Additionally, since you mentioned lactose sensitivity, Havarti is often easier to digest for those with mild lactose intolerance, but always check the packaging for specific lactose content.

To make the grilled cheese, use whole-grain bread for added fiber and nutrients. Cook the sandwich on medium heat to ensure the cheese melts evenly without burning the bread. Enjoy your delicious, kid-friendly meal!

In [87]:
# @title Step 6: Tool Call Outputs
print("\n" + "="*70)
print("Step 6: With Tool Call Outputs")
print("="*70)

# Simulated tool functions: Results from function calls (APIs, databases, calculations)
check_grocery_inventory = {
    "Kraft American Singles": {"price": 4.99, "in_stock": True},
    "Whole Foods Mild Cheddar": {"price": 13.49, "in_stock": True},
    "Tillamook Medium Cheddar": {"price": 55.99, "in_stock": True},
    "Boar's Head Vermont Cheddar": {"price": 8.99, "in_stock": False}
}
check_lactose_content = {
    "american": {"lactose_per_slice": "0.1g", "note": "processed, lactose removed"},
    "mild_cheddar": {"lactose_per_slice": "0.2g", "note": "naturally low"},
    "aged_cheddar": {"lactose_per_slice": "0.1g", "note": "aging reduces lactose"}
}
simulate_melting = {
    "american": "Smooth melt, no separation, kid-friendly texture",
    "mild_cheddar": "Good melt, slight oil separation at high heat, richer flavor"
}

# Format tool outputs for context
tool_outputs = f"""Tool Call Results:

Grocery Inventory at Whole Foods:
{str(check_grocery_inventory)}

Lactose Content Analysis:
{str(check_lactose_content)}

Melting Simulation Results:
{str(simulate_melting)}"""

# System prompt
system_prompt = f"""You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

{memory_context}

Use this information to personalize your recommendations."""

# Inject tool outputs into the prompt
enhanced_user_prompt = f"""{str(tool_outputs)}
Use this real-time data to make your recommendation.
User question: {user_prompt}"""

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": enhanced_user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 6: With Tool Call Outputs


For a grilled cheese sandwich that aligns with your preferences and constraints, I recommend using the Whole Foods Mild Cheddar. Here's why:

1. **Lactose Sensitivity**: Mild cheddar is naturally low in lactose, with approximately 0.2g per slice, making it a suitable choice for those with lactose sensitivity.

2. **Natural Ingredients**: Whole Foods Mild Cheddar is likely to meet your requirement for simple, natural ingredients without artificial additives, as Whole Foods typically emphasizes clean ingredient lists.

3. **Melting Properties**: Mild cheddar has a good melting profile, providing a rich flavor with a slight oil separation at high heat. This can be managed by cooking at a moderate temperature to achieve a smooth, creamy texture.

4. **Flavor Profile**: The mild cheddar offers a richer flavor compared to processed cheeses, which can enhance the taste of your grilled cheese without being too sharp for kids.

5. **Availability and Budget**: Priced at $13.49, it fits within your budget and is available at Whole Foods, where you shop.

To make the perfect grilled cheese, use a non-stick skillet over medium heat. Butter the outside of the bread slices for a golden crust, and cook the sandwich slowly to allow the cheese to melt thoroughly without burning the bread. This approach will ensure a delicious, kid-friendly meal that respects your dietary needs and ingredient preferences.

In [88]:
# @title Step 7: Structured Output
print("\n" + "="*70)
print("Step 7: With Structured Output")
print("="*70)

# 🔒 JSON Schema for strict output
schema = {
    "type": "object",
    "properties": {
        "top_recommendation": {"type": "string"},
        "alternatives": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1
        },
        "cooking_tips": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1
        },
        "why_this_works": {"type": "string"}
    },
    "required": ["top_recommendation", "alternatives", "cooking_tips", "why_this_works"],
    "additionalProperties": False
}

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": enhanced_user_prompt}
    ],
    # ✅ Structured Outputs with strict JSON Schema
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "CheeseAdvice",
            "strict": True,          # guarantees valid JSON and schema adherence
            "schema": schema
        }
    },
    temperature=0.2,
)

# The model returns a JSON string in message.content
raw_json = response.choices[0].message.content
data = json.loads(raw_json)

# Pretty print as fenced JSON (or use it programmatically)
display(Markdown(f"```json\n{json.dumps(data, indent=2)}\n```"))


Step 7: With Structured Output


```json
{
  "top_recommendation": "Whole Foods Mild Cheddar",
  "alternatives": [
    "Kraft American Singles"
  ],
  "cooking_tips": [
    "Use a non-stick pan to prevent sticking and ensure even browning.",
    "Cook on medium-low heat to allow the cheese to melt thoroughly without burning the bread.",
    "Butter the outside of the bread for a crispy, golden crust."
  ],
  "why_this_works": "Whole Foods Mild Cheddar is a great choice because it has a natural ingredient list, is naturally low in lactose, and provides a rich flavor that kids enjoy. It melts well, making it suitable for grilled cheese, and fits within the budget. While it may have slight oil separation at high heat, cooking at medium-low heat can mitigate this issue."
}
```

# **Tools, MCP (Model Context Protocol)**

In [ ]:
# Define the LLM model and question
model = "gpt-4o"
user_prompt = "What's the best cheese for a grilled cheese sandwich?"

## **Tools**

Let us build a simple tool that returns random fun facts from Pavlos.

In [91]:
# --- Fun Facts data ---
PAVLOS_FUN_FACTS = [
    "Mozzarella is the most consumed cheese in the U.S., largely due to pizza.",
    "Gruyère and Emmental are classics for fondue thanks to their smooth melt.",
    "Aging boosts sharpness; 12-month cheddar has nuttier, deeper flavors.",
    "High-moisture cheeses (young gouda, fontina, jack) melt especially evenly.",
    "Rind-on bries are edible; the rind adds mushroomy, earthy notes.",
    "Salt helps control moisture and rind formation during cheesemaking.",
    "Browned spots on grilled cheese = Maillard reaction (not caramelization).",
    "American slices are engineered to melt at lower temperatures.",
    "Fresh cheeses (ricotta, chèvre) soften but don’t stretch like mozzarella.",
    "Taleggio’s washed rind brings savory depth and melts well.",
]

def pavlos_fun_fact():
  return {
      "fact": random.choice(PAVLOS_FUN_FACTS),
      "generated_at": int(time.time())
  }

# ✅ Define a Tool
@function_tool(name_override="pavlos_fun_fact_tool")
async def pavlos_fun_fact_tool() -> dict:
  """Return one random fun fact from Pavlos about cheese. Call multiple times if you want several facts."""
  return pavlos_fun_fact()

# Test tool
print(pavlos_fun_fact())
print(pavlos_fun_fact())
print(pavlos_fun_fact())

{'fact': 'Gruyère and Emmental are classics for fondue thanks to their smooth melt.', 'generated_at': 1790132526}
{'fact': 'Mozzarella is the most consumed cheese in the U.S., largely due to pizza.', 'generated_at': 1790132526}
{'fact': 'Taleggio’s washed rind brings savory depth and melts well.', 'generated_at': 1790132526}


In [92]:
# System Prompt
system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

Call the pavlos_fun_fact_tool tool 2–3 times and add facts to your response."""

# Build an agent that uses "pavlos_fun_fact_tool"
culinary_expert_agent = Agent(
    name="Culinary Expert",
    instructions=system_prompt,
    model=model,
    tools=[pavlos_fun_fact_tool],
    model_settings=ModelSettings(temperature=0.2),
)

print("\n" + "="*50)
print("Culinary Expert Agent (with fun facts tool call):",model)
print("="*50)
agent_result = await Runner.run(culinary_expert_agent, input=user_prompt)
print(f"Final Response:")
display(Markdown(agent_result.final_output))


Culinary Expert Agent (with fun facts tool call): gpt-4o
I am a tool
I am a tool
I am a tool
Final Response:


For a perfect grilled cheese sandwich, consider using a combination of cheeses to achieve the ideal melt and flavor. Here are some top choices:

1. **American Cheese**: Known for its excellent melting properties, American cheese provides a creamy texture and mild flavor. It's a classic choice for grilled cheese and widely available in grocery stores.

2. **Cheddar**: Opt for a medium or sharp cheddar for a more pronounced flavor. Cheddar melts well, especially when combined with a higher moisture cheese. It adds a nice tang to the sandwich.

3. **Fontina**: This cheese is known for its smooth melting qualities due to its high moisture content. It has a mild, nutty flavor that complements other cheeses well.

4. **Gouda**: Young gouda is another high-moisture cheese that melts evenly. It offers a slightly sweet and creamy taste, enhancing the overall flavor profile of your sandwich.

### Practical Tips:
- **Combination**: Mix American cheese with cheddar or gouda to balance meltability and flavor.
- **Temperature**: Cook your sandwich on medium heat to ensure the cheese melts thoroughly without burning the bread.
- **Butter**: Spread butter on the outside of the bread for a crispy, golden crust.

### Fun Facts:
- Salt plays a crucial role in cheesemaking by controlling moisture and rind formation.
- High-moisture cheeses like young gouda and fontina melt especially evenly, making them ideal for grilled cheese.

These choices and tips will help you craft a delicious, gooey grilled cheese sandwich with a satisfying flavor and texture. Enjoy!

In [93]:
# If you want to list all the tools called in the agent run
# Tools Called
tool_calls = [it for it in agent_result.new_items if isinstance(it, ToolCallItem)]
print("Tools called:")
for i, it in enumerate(tool_calls, 1):
    raw = it.raw_item
    name = getattr(raw, "name", getattr(raw, "tool_name", "unknown_tool"))
    args = getattr(raw, "arguments", getattr(raw, "input", None))
    print(f"{i}. {name}  args={args}")

# Tools Output
tool_outputs = [it for it in agent_result.new_items if isinstance(it, ToolCallOutputItem)]
print("\nTool outputs:")
for i, it in enumerate(tool_outputs, 1):
    raw = it.raw_item
    print(f"{i}. {it.output!r}")

Tools called:
1. pavlos_fun_fact_tool  args={}
2. pavlos_fun_fact_tool  args={}
3. pavlos_fun_fact_tool  args={}

Tool outputs:
1. {'fact': 'Salt helps control moisture and rind formation during cheesemaking.', 'generated_at': 1790132531}
2. {'fact': 'High-moisture cheeses (young gouda, fontina, jack) melt especially evenly.', 'generated_at': 1790132531}
3. {'fact': 'Salt helps control moisture and rind formation during cheesemaking.', 'generated_at': 1790132531}


## **MCP (Model Context Protocol)**

## **Using Tools from an MCP Server**

In a real use case we would not have to build the tool ourselves. In our examples let's say Pavlos maintains his cheese fun facts in a PostgreSQL database. Traditionally if we need to integrate with the fun facts database, he would give us access to the database or create some API for us.

Like we saw in lecture, the limitation of this approach is we have to build custom integrations for every tool. This is why we use the concept of MCP. Pavlos will build an MCP Server and give us the link to the server.

In [95]:
# MCP Server
pavlos_mcp_url = "https://pavlos-fun-facts.dlops.io/mcp"

async def run_agent(query: str):
  async with AsyncExitStack() as stack:
      # Build a list of MCP servers we want to use
      mcp_servers = []

      # Connect to the Pavlos Cheese MCP server for this session
      mcp_server = await stack.enter_async_context(
          MCPServerStreamableHttp(
              name="Pavlos FastMCP",
              params={"url": pavlos_mcp_url},          # MCP endpoint
              cache_tools_list=True,                   # optional: cache tool discovery
              max_retry_attempts=3,
              client_session_timeout_seconds=180,
          )
      )
      mcp_servers.append(mcp_server)

      # Agent that uses MCP for tool use
      agent = Agent(
          name="Culinary Expert (MCP)",
          instructions=system_prompt,
          model=model,
          # IMPORTANT: add the server in mcp_servers (not tools)
          mcp_servers=mcp_servers,
          model_settings=ModelSettings(temperature=0.2, tool_choice="auto"),
      )
      result = await Runner.run(agent, input=query)
      return result.final_output

# Call it:
out = await run_agent(user_prompt)
from IPython.display import Markdown, display
display(Markdown(out))

For the perfect grilled cheese sandwich, consider using a combination of cheeses to achieve the ideal melt and flavor. Here are some top choices:

1. **Cheddar**: 
   - **Melting Properties**: Cheddar melts beautifully, especially when aged for about 3-6 months.
   - **Flavor Profile**: Offers a sharp, tangy flavor that pairs well with bread.
   - **Availability**: Widely available in most grocery stores.
   - **Cooking Tip**: Use medium heat to ensure even melting without burning the bread.

2. **Gruyère**:
   - **Melting Properties**: Known for its smooth melt, making it perfect for a creamy texture.
   - **Flavor Profile**: Nutty and slightly sweet, adding depth to your sandwich.
   - **Availability**: Found in most cheese sections, especially in larger stores.
   - **Cooking Tip**: Combine with Cheddar for a balanced flavor and melt.

3. **Mozzarella**:
   - **Melting Properties**: Offers a gooey, stretchy melt.
   - **Flavor Profile**: Mild, allowing other flavors to shine.
   - **Availability**: Easily found in both block and shredded forms.
   - **Cooking Tip**: Pair with a sharper cheese like Cheddar for added flavor.

4. **Brie**:
   - **Melting Properties**: Soft and creamy, melts quickly.
   - **Flavor Profile**: Rich and buttery with earthy notes from the rind.
   - **Availability**: Available in most grocery stores, often in the specialty cheese section.
   - **Cooking Tip**: Use rind-on for added flavor; it contributes mushroomy, earthy notes.

### Fun Cheese Facts:
- **Salt's Role**: Salt helps control moisture and rind formation during cheesemaking, crucial for texture and flavor.
- **Fondue Favorites**: Gruyère and Emmental are classics for fondue thanks to their smooth melt, making them great for grilled cheese too.
- **Brie Rind**: Rind-on bries are edible; the rind adds mushroomy, earthy notes, enhancing the overall taste.

By combining these cheeses, you can create a grilled cheese sandwich that's both flavorful and perfectly melted. Enjoy experimenting with different combinations to find your favorite!

## **Build your own MCP Server** (Optional)

This section is optional where we can build and host our own MCP. Here we use the Python framework FastMCP to build an MCP server.

In [104]:
# --- Define your MCP server with one tool: pavlos_fun_fact
mcp = FastMCP("Pavlos Cheese MCP Local")

# ✅ Define a Tool as MCP Server
@mcp.tool
def pavlos_fun_fact_mcp() -> dict:
    """Return one random fun fact from Pavlos about cheese. Call multiple times if you want several facts."""
    print("Pavlos Fun Fact Tool Called")
    return pavlos_fun_fact()

# 🏃‍♀️ Run the server over HTTP on port 8000 in a background thread
def run_server():
    # HTTP transport serves the MCP endpoint at /mcp
    mcp.run(transport="http", host="127.0.0.1", port=8001)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("FastMCP server starting on http://127.0.0.1:8001/mcp")

print("Server thread alive:", thread.is_alive())

FastMCP server starting on http://127.0.0.1:8000/mcp
Server thread alive: True


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 4.0.5                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                🖥  Server:      Pavlos Cheese MCP Local, 4.0.5                │                  
                 │                🚀 Deploy free: https://horizon.prefect.io                    │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

Next we will use this local MCP server as the source of our Tool.

In [106]:
# MCP Server
pavlos_mcp_url = "http://127.0.0.1:8001/mcp"

async def run_agent(query: str):
  async with AsyncExitStack() as stack:
      # Build a list of MCP servers we want to use
      mcp_servers = []

      # Connect to the Pavlos Cheese MCP server for this session
      mcp_server = await stack.enter_async_context(
          MCPServerStreamableHttp(
              name="Pavlos FastMCP",
              params={"url": pavlos_mcp_url},
              cache_tools_list=True,
              max_retry_attempts=3,
          )
      )
      mcp_servers.append(mcp_server)

      # Agent that uses MCP for tool use
      agent = Agent(
          name="Culinary Expert (MCP)",
          instructions=system_prompt,
          model=model,
          # IMPORTANT: add the server in mcp_servers (not tools)
          mcp_servers=mcp_servers,
          model_settings=ModelSettings(temperature=0.2, tool_choice="auto"),
      )
      result = await Runner.run(agent, input=query)
      return result.final_output

# Call it:
out = await run_agent(user_prompt)
from IPython.display import Markdown, display
display(Markdown(out))

INFO:     127.0.0.1:49802 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49802 - "POST /mcp HTTP/1.1" 200 OK
Pavlos Fun Fact Tool Called
INFO:     127.0.0.1:49802 - "POST /mcp HTTP/1.1" 200 OK
Pavlos Fun Fact Tool Called
INFO:     127.0.0.1:49802 - "POST /mcp HTTP/1.1" 200 OK
Pavlos Fun Fact Tool Called
INFO:     127.0.0.1:49802 - "POST /mcp HTTP/1.1" 200 OK


For the perfect grilled cheese sandwich, consider using a combination of cheeses to achieve the ideal melt and flavor:

1. **American Cheese**: Known for its smooth melting properties, American cheese is engineered to melt at lower temperatures, making it a classic choice for grilled cheese. It provides a creamy texture and mild flavor that many people love.

2. **Cheddar**: A sharp cheddar adds a robust flavor to your sandwich. It melts well, especially when combined with other cheeses. Opt for a medium or sharp cheddar to balance the flavors.

3. **Gruyère**: This cheese is famous for its smooth melt, similar to its use in fondue. Gruyère adds a nutty and slightly sweet flavor, enhancing the overall taste of your sandwich.

4. **Fontina**: High-moisture cheeses like fontina melt especially evenly, providing a creamy texture. Fontina has a mild, buttery flavor that complements other cheeses well.

### Practical Tips:
- **Combination**: Try mixing American cheese with cheddar or Gruyère for a balanced flavor and perfect melt.
- **Temperature**: Cook your sandwich on medium-low heat to ensure the cheese melts thoroughly without burning the bread.
- **Bread Choice**: Use a hearty bread like sourdough or a classic white bread for the best texture.

### Fun Facts:
- Gruyère and Emmental are classics for fondue thanks to their smooth melt.
- American slices are engineered to melt at lower temperatures.
- High-moisture cheeses like young gouda, fontina, and jack melt especially evenly.

These combinations and tips will help you create a deliciously gooey and flavorful grilled cheese sandwich!

# **Multi-Agent System: The Cheese Quiz Guild 🧀**

We will build the Cheese Quiz Guild 🧀 with the following requirements:

*   A user can request a quiz on any specific cheese topic  (e.g., "French Cheeses," "Hard Cheeses," "Cheeses from Star Wars")
*   The Guild of AI agents must work together to research, write, verify, and format a high-quality quiz.
* This implementation will illustrate the core principles of AI agents: specialized roles, tool usage, and collaborative workflows.


First we will define some utilities required to build The Cheese Quiz Guild Agents.

In [ ]:
# @title Some Utils

SCHEMA_PATH = Path("quiz-schema.json")

with open("quiz-ui.html", "r") as f:
    QUIZ_UI = f.read()

class QuizAgentOutputSchema(AgentOutputSchemaBase):
    """
    Custom output schema that:
      - returns your JSON Schema (for Structured Outputs)
      - validates and parses the model's JSON into a Python object
    """

    def __init__(self, schema_path: Path = SCHEMA_PATH):
        self._schema_path = schema_path
        self._schema: Dict[str, Any] = json.loads(
            self._schema_path.read_text(encoding="utf-8")
        )
        # tip: assert top-level shape matches Structured Outputs support
        # (only supported/strict JSON Schema features) per docs:
        # https://platform.openai.com/docs/guides/structured-outputs/supported-schemas
        # The uploaded file uses only supported constructs (object/array/string/integer).

    # --- AgentOutputSchemaBase methods required by the SDK ---
    def is_plain_text(self) -> bool:
        return False

    def name(self) -> str:
        return "QuizJSON"

    def json_schema(self) -> Dict[str, Any]:
        # The SDK will pass this to the model as the response schema.
        return self._schema

    def is_strict_json_schema(self) -> bool:
        # Opt into strict mode so the SDK enforces the subset supported by Structured Outputs.
        return True

    def validate_json(self, json_str: str) -> Any:
        """
        Must return the validated object or raise ModelBehaviorError.
        """
        try:
            obj = json.loads(json_str)
        except Exception as e:
            raise ModelBehaviorError(f"Invalid JSON: {e}") from e

        try:
            js_validate(instance=obj, schema=self._schema)  # runtime guardrail
        except ValidationError as e:
            # Signal to the SDK that the model misbehaved.
            raise ModelBehaviorError(
                f"JSON failed schema validation: {e.message}"
            ) from e

        return obj


def render_quiz_html(
    quiz: Dict[str, Any],
    *,
    title: str | None = None,
    shuffle_questions: bool = False,
    shuffle_options: bool = False,
) -> str:

    topic = title or quiz.get("topic") or "Quiz"

    norm_questions = []
    for idx, q in enumerate(quiz.get("questions", []), start=1):
        # Try both shapes
        q_text = q.get("question") or q.get("text") or ""
        opts = q.get("choices") or q.get("options") or []
        # Keys/letters
        normalized_opts = []
        for opt in opts:
            key = opt.get("key") or opt.get("letter") or ""
            normalized_opts.append({"key": str(key), "text": opt.get("text", "")})
        answer_key = q.get("answer_key") or q.get("answer") or ""
        norm_questions.append(
            {
                "id": q.get("id") or idx,
                "text": q_text,
                "options": normalized_opts,
                "answer": str(answer_key),
            }
        )

    client_payload = {
        "title": topic,
        "questions": norm_questions,
        "settings": {
            "shuffleQuestions": bool(shuffle_questions),
            "shuffleOptions": bool(shuffle_options),
        },
    }

    # Escape + serialize once for safe embedding in a <script> tag
    data_json = json.dumps(client_payload, ensure_ascii=False)

    return QUIZ_UI.replace("{ data_json }", data_json)


# ───────────────────────────
# Shared types
# ───────────────────────────
class CheeseFact(BaseModel):
    statement: str
    source: Optional[str] = None

class QuizChoice(BaseModel):
    key: str
    text: str


class QuizQ(BaseModel):
    question: str
    choices: List[QuizChoice]
    answer_key: str
    source_note: Optional[str] = None


class QuizDraft(BaseModel):
    topic: str
    questions: List[QuizQ]


class ReviewResult(BaseModel):
    approved: bool
    notes: str = ""
    needs_revision_indices: List[int] = Field(default_factory=list)
    needs_user_opinion: bool = False


@dataclass
class GuildContext:
    fact_cache: dict[str, List[CheeseFact]]


@dataclass
class OrchestratorResult:
    response_text: str
    structured_json: dict
    formatted_md: str


# ───────────────────────────
# Helper tools
# ───────────────────────────


@function_tool
def ensure_unique_answers(question: str, choices: List[str]) -> bool:
    norm = [c.strip().lower() for c in choices]
    return len(set(norm)) == len(norm)


@function_tool
def ask_head_cheesemonger(prompt: str, default_decision: bool = True) -> bool:
    # In real UI, ask the user. Here we simulate.
    return default_decision

## **Meet the Agents: The Guild Members**

In this section we build the core agents required for our multi-agent system. Giving our agents distinct personas will make them more specialized in their roles.

#### 🧐 **Professor Stilton, the Fact-Finder**
This agent's primary job is to take the user's topic and use a **search tool** to find interesting, verifiable facts.

#### 🌱 **Cashew, the Plant-Based Pro**
Cashew supplements Professor Stilton's research with up to three additional facts when the topic involves vegan or plant-based cheese.

#### ✍️ **Brie, the Creative Wordsmith**
The Brie Agent takes the raw facts from Professor Stilton and transforms them into engaging quiz questions (multiple choice, true/false, etc.). She's the core "creative" LLM.

#### 🧀 **Cheddar, the Sharp Critic**
Cheddar Agent reviews the questions and answers created by Brie. He checks them against Stilton's original facts for accuracy. He also ensures the questions are clear and not too easy or impossibly hard.

#### 🎨 **Mozzarella, the Master Formatter**
The Mozzarella Agent takes a quiz that is approved by Cheddar and converts it to Markdown format.

#### 🎨 **Ricotta, the Schema Scribe**
Ricotta takes a quiz that is approved by Cheddar and converts it into strict JSON.

In [110]:
# Professor Stilton, the Fact-Finder
stilton_agent = Agent(
    name="Professor Stilton, the Fact-Finder",
    model="gpt-4o",
    handoff_description="Gathers concise, citable cheese facts for a topic.",
    instructions=(
        "You are Professor Stilton. Given a cheese topic, use web search to gather ~5 concise, "
        "verifiable facts suitable for quiz questions. Include short source notes (domain or title). "
    ),
    tools=[WebSearchTool()],
    output_type=List[CheeseFact],
    model_settings=ModelSettings(temperature=0.3, tool_choice="auto"),
)

# Test
result = await Runner.run(stilton_agent, "Cheese topic: brie")
cheese_facts = result.final_output
for fact in cheese_facts:
    print(f"{fact.statement} ({fact.source})")

Brie is a soft, cow’s‑milk cheese named after the Brie region in northern France, with its name deriving from the Gaulish word “briga,” meaning “hill” or “height.” (Wikipedia (Brie) & Online Etymology Dictionary ([en.wikipedia.org](https://en.wikipedia.org/wiki/Brie?utm_source=openai)))
Brie has been produced in the town of Meaux since at least the 8th century and was historically known as the “Queen’s cheese” or “queen of cheeses.” (Wikipedia (Brie) ([en.wikipedia.org](https://en.wikipedia.org/wiki/Brie?utm_source=openai)))
Only two varieties—Brie de Meaux and Brie de Melun—hold protected status (AOC/AOP), with Brie de Meaux requiring raw cow’s‑milk, a minimum four‑week aging, and a large wheel of about 36–37 cm in diameter. (Wikipedia (Brie) & INAO & French Agriculture Ministry ([en.wikipedia.org](https://en.wikipedia.org/wiki/Brie?utm_source=openai)))
Brie typically contains between 60 % and 75 % butterfat, giving it a rich, creamy texture and flavor notes described as creamy, nutty

In [111]:
# Brie, the Creative Wordsmith
brie_agent = Agent(
    name="Brie, the Creative Wordsmith",
    model="gpt-4o",
    handoff_description="Writes engaging multiple-choice questions from facts.",
    instructions=(
        "Turn the given facts into exactly 5 multiple-choice questions. Each has 4 options (keys a,b,c,d) "
        "with one correct answer. Make them fun but precise; avoid trickiness; include source_note if present. "
        "Return ONLY structured QuizDraft."
    ),
    output_type=QuizDraft,
    model_settings=ModelSettings(temperature=0.7),
)

# Test
input = "Facts:\n" + "\n".join(f"- {f.statement}" for f in cheese_facts)
result = await Runner.run(brie_agent, input=input)
quiz_draft = result.final_output
display(quiz_draft)

QuizDraft(topic='Brie Cheese', questions=[QuizQ(question="What does the name 'Brie' derive from?", choices=[QuizChoice(key='a', text='A French village'), QuizChoice(key='b', text='A type of cow'), QuizChoice(key='c', text="A Gaulish word for 'hill'"), QuizChoice(key='d', text='A type of wine')], answer_key='c', source_note=None), QuizQ(question='Which town has been producing Brie since at least the 8th century?', choices=[QuizChoice(key='a', text='Paris'), QuizChoice(key='b', text='Meaux'), QuizChoice(key='c', text='Lyon'), QuizChoice(key='d', text='Reims')], answer_key='b', source_note=None), QuizQ(question='What is required for Brie de Meaux to hold its protected status?', choices=[QuizChoice(key='a', text='Pasteurized cow’s-milk and a two-week aging period'), QuizChoice(key='b', text='Raw cow’s-milk and a four-week aging period'), QuizChoice(key='c', text='Raw goat’s-milk and a five-week aging period'), QuizChoice(key='d', text='Pasteurized cow’s-milk and a large wheel')], answer_ke

In [112]:
# Cheddar, the Sharp Critic
cheddar_agent = Agent(
    name="Cheddar, the Sharp Critic",
    handoff_description="Reviews quiz clarity & correctness vs facts; requests fixes.",
    instructions=(
        "Compare each question to the supplied facts. Flag ambiguous, too-easy/too-hard, or ungrounded items. "
        "Use ensure_unique_answers to detect duplicate options. If borderline difficulty or taste-based, "
        "set needs_user_opinion=True. Produce ReviewResult with indices & notes."
    ),
    tools=[ensure_unique_answers, ask_head_cheesemonger],
    output_type=ReviewResult,
    model_settings=ModelSettings(temperature=0.2),
)

# Mozzarella, the Master Formatter
mozzarella_agent = Agent(
    name="Mozzarella, the Master Formatter",
    handoff_description="Formats the approved quiz for delivery.",
    instructions=(
        "Format the final, approved QuizDraft into Markdown:\n"
        "• Title\n• Numbered questions\n• Lettered options (a–d)\n• **Answer Key**\n• Optional sources."
    ),
    output_type=str,
    model_settings=ModelSettings(temperature=0.2),
)

# Cashew Agent, Dynamic specialist
cashew_agent = Agent(
    name="Cashew, the Plant-Based Pro",
    handoff_description="Adds vegan/plant-based cheese expertise to research.",
    instructions=(
        "Augment Stilton with up to 3 extra facts when the topic includes vegan/plant-based."
    ),
    tools=[WebSearchTool()],
    output_type=List[CheeseFact],
    model_settings=ModelSettings(temperature=0.4),
)

# Ricotta Agent
ricotta_agent = Agent(
    name="Ricotta, the Schema Scribe",
    handoff_description="Converts an approved quiz into strict JSON that matches the provided schema.",
    instructions=(
        "You will receive an approved quiz (questions with options and the correct answer). "
        "Output ONLY valid JSON that conforms to the provided schema. No markdown, no comments.\n"
        "RULES:\n"
        "1) Output ONLY raw JSON (no Markdown fences, no commentary).\n"
        "2) Use sequential integer IDs starting at 1.\n"
        "3) Each question MUST have exactly four options with 'letter' keys: 'a', 'b', 'c', 'd'.\n"
        "4) The 'answer' must equal one of the option letters ('a'|'b'|'c'|'d').\n"
        "5) Preserve the original question wording and option texts; do not invent facts.\n"
        "6) Do not include sources or extra fields—only the fields in the schema.\n"
    ),
    output_type=QuizAgentOutputSchema(),  # <- custom schema object
    model_settings=ModelSettings(temperature=0.1),
)

## **Define Tools**

In this section we will create agents as tools for orchestration.

In [113]:
# Agents-as-tools for orchestration
stilton_tool = stilton_agent.as_tool(
    "research_facts", "Find ~5 concise, citable facts for a topic."
)
brie_tool = brie_agent.as_tool("write_quiz", "Convert facts into a 5-question MCQ quiz.")
cheddar_tool = cheddar_agent.as_tool("review_quiz", "QA the quiz against the facts.")
mozzarella_tool = mozzarella_agent.as_tool("format_quiz", "Format quiz into Markdown.")
ricotta_tool = ricotta_agent.as_tool("format_quiz_json", "Format quiz into JSON.")
cashew_tool = cashew_agent.as_tool(
    "vegan_specialist", "Supplement with vegan cheese facts as needed."
)

## **Agent Orchestrator**

In this section we build the workflow that assembles all the agents together.

1.  **The Order (Input)**: The user requests a quiz on "Italian Cheeses."
2.  **Research (Agent 1)**: The request goes to **Professor Stilton**. He searches for 5 key facts about Pecorino, Parmesan, Mozzarella, etc.
3.  **Creation (Agent 2)**: The facts are passed to **Brie**. She crafts 5 multiple-choice questions based on the facts.
4.  **Verification (Agent 3)**: The draft quiz is sent to **Cheddar**. He flags one question as potentially confusing and sends it back to Brie with a note. Brie revises it and sends it back to Cheddar, who now approves the whole set.
5.  **Formatting (Agent 4)**: The final, approved quiz goes to **Ricotta**, who formats it into structured JSON.
6.  **Delivery (Output)**: The final, polished quiz is presented to the user.


In [114]:
orchestrator = Agent(
    name="Cheese Quiz Guild Orchestrator",
    instructions=(
        "Act as the head of the Cheese Quiz Guild. "
        "Follow the assembly line strictly:\n"
        "1) Call research_facts(topic). If the topic is vegan/plant-based, also call vegan_specialist.\n"
        "   Merge/trim to ~5 total facts (no duplicates), and update context.fact_cache[topic].\n"
        "2) Call write_quiz(topic, facts).\n"
        "3) Call review_quiz(draft, facts). If not approved and not needs_user_opinion, "
        "   send exactly one revision request back to write_quiz and then re-review. "
        "   (Make at most 2 total review rounds.)\n"
        "   If needs_user_opinion, call ask_head_cheesemonger to decide and proceed.\n"
        "4) format_quiz_json(approved_draft) → JSON.\n"
        "Keep outputs structured between steps; only the last step returns JSON."
    ),
    tools=[
        stilton_tool,
        cashew_tool,
        brie_tool,
        cheddar_tool,
        ricotta_tool,
        ask_head_cheesemonger,
    ],
    output_type=QuizAgentOutputSchema(),
    model_settings=ModelSettings(),
)

In [115]:
async def run_guild(topic: str) -> str:

    # seed the orchestrator with the user's topic
    prompt = f"Customer requests a quiz on: {topic}. Produce the final JSON."
    result = await Runner.run(
        orchestrator, prompt, max_turns=12
    )
    quiz = result.final_output
    html_str = render_quiz_html(quiz, shuffle_questions=False, shuffle_options=False)
    return HTML(html_str)

# **Cheese Quiz Guild Agent in Action**

The Cheese Quiz Guild Agent coordinates specialized AI agents to collaboratively research, draft, verify, and format a high-quality cheese trivia quiz on any given topic.

## Example 1:

In [116]:
# Create a quiz
await run_guild("Cheeses from Star Wars")

ERROR:openai.agents:Error getting response
ERROR:openai.agents:Error getting response
ERROR:openai.agents:Error getting response


## Example 2:

In [ ]:
# Create a quiz
await run_guild("Greek cheeses")

ERROR:openai.agents:Error getting response
ERROR:openai.agents:Error getting response
ERROR:openai.agents:Error getting response
